# Projecto Zuber: Análise de Transporte em Chicago

## 🛠️ Passo 1: Ingestão e Auditoria de Dados (Check-in)

O objetivo desta fase é carregar as bases de dados extraídas via SQL e garantir que a estrutura técnica está pronta para análise, mitigando riscos de corrupção de tipos ou valores ausentes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Importa├º├úo
df_companies = pd.read_csv('C:/moved_project_sql_result_01.csv')
df_neighborhoods = pd.read_csv('C:/moved_project_sql_result_04.csv')
df_hypothesis = pd.read_csv('C:/moved_project_sql_result_07.csv')

In [ ]:
df_companies.info()

In [ ]:
df_companies.head()

In [ ]:
df_companies.isna().sum()

In [ ]:
df_companies.duplicated().sum()

In [ ]:
df_companies.describe()

In [ ]:
df_neighborhoods.info()

In [ ]:
df_neighborhoods.head()

In [ ]:
df_neighborhoods.duplicated().sum()

In [ ]:
df_neighborhoods.isna().sum()

In [ ]:
df_neighborhoods.describe()

In [ ]:
def audit_data(df, name):
    print(f"--- Auditoria: {name} ---")
    print(f"Dimens├Áes: {df.shape}")
    print("\nTipos de Dados e Nulos:")
    print(df.info())
    print("\nValores Duplicados:", df.duplicated().sum())
    print("\nEstat├¡stica Descritiva:")
    print(df.describe())
    print("-" * 30, "\n")
# Executando a auditoria inicial
audit_data(df_companies, "Empresas")
audit_data(df_neighborhoods, "Bairros (Destinos)")
audit_data(df_hypothesis, "Teste de Hip├│tese")

### 🧹 Passo 2: Preparação e Limpeza dos Dados

In [ ]:
# --- LIMPEZA  ---

# 1. Convers├úo de data para o dataset de hip├│tese 
df_hypothesis['start_ts'] = pd.to_datetime(df_hypothesis['start_ts'])

# 2. Verifica├º├úo de valores negativos 
if (df_companies['trips_amount'] < 0).any():
    print("ALERTA: Detectados valores negativos em trips_amount. Removendo...")
    df_companies = df_companies[df_companies['trips_amount'] >= 0]

### 📊 Fase 3: Identificação dos 10 Principais Bairros

In [ ]:
#  10 principais destinos
top_10_neighborhoods = df_neighborhoods.sort_values(by='average_trips', ascending=False).head(10)

print("TOP 10 DESTINOS (NOVEMBRO 2017):")
display(top_10_neighborhoods)

### 📈 Fase 4: Visualização e Conclusões Técnicas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.barplot(data=df_companies.sort_values(by='trips_amount', ascending=False).head(10), 
            x='trips_amount', y='company_name', hue='company_name', palette='Blues_d', legend=False)
plt.title('Top 10 Empresas por Volume de Corridas (15-16 Nov 2017)')
plt.show()

Conclusão: Existe uma discrepância enorme entre o líder (Flash Cab) e os demais. Isso indica que o mercado de Chicago é dominado por grandes frotas, dificultando a entrada de novos players (como a Zuber) a menos que haja um diferencial de preço ou tempo de espera.

Gráfico 2: Densidade Geográfica (Destinos)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_neighborhoods, x='average_trips', y='dropoff_location_name', hue='dropoff_location_name', palette='Reds_d', legend=False)
plt.title('Top 10 Bairros por M├®dia de Viagens Finalizadas (Nov 2017)')
plt.show()

Conclus├úo: O Loop e River North n├úo s├úo apenas destinos populares; eles s├úo outliers de demanda. A Zuber deve focar 80% de sua estrat├®gia de posicionamento de motoristas nestas zonas durante o hor├írio comercial.

### 🧪 Passo 5: Teste de Hipótese (Estatística de Produção)

In [ ]:
import pandas as pd
from scipy import stats as st

# 1. Carga dos dados 
df_trips = pd.read_csv('C:/moved_project_sql_result_07.csv')

# 2. Segmenta├º├úo das Amostras para Teste
# Sele├º├úo de dura├º├Áes (em segundos) para as duas condi├º├Áes meteorol├│gicas
rainy_saturdays = df_trips[df_trips['weather_conditions'] == 'Bad']['duration_seconds']
good_saturdays = df_trips[df_trips['weather_conditions'] == 'Good']['duration_seconds']

# 3. Formula├º├úo das Hip├│teses
# H0 (Nula): A dura├º├úo m├®dia das viagens do Loop para o O'Hare ├® a mesma em s├íbados chuvosos e bons.
# H1 (Alternativa): A dura├º├úo m├®dia das viagens do Loop para o O'Hare muda nos s├íbados chuvosos.

# 4. Defini├º├úo do N├¡vel de Signific├óncia (Alpha)
# Defini├º├úo alpha em 0.05 (5%), .
alpha = 0.05 

# 5. Auditoria de Vari├óncia (Teste de Levene)
stat_levene, p_levene = st.levene(rainy_saturdays, good_saturdays)
equal_var_bool = p_levene > 0.05

# 6. Execu├º├úo do Teste t de Student para duas amostras independentes
results = st.ttest_ind(rainy_saturdays, good_saturdays, equal_var=equal_var_bool)

print(f"P-value: {results.pvalue}")

# 7.  Decis├úo Estat├¡stica
if results.pvalue < alpha:
    print("Conclus├úo: Rejeita a hip├│tese nula.")
else:
    print("Conclus├úo: N├úo podemos rejeitar a hip├│tese nula.")

# Análise Estratégica – Entrada da Zuber em Chicago

## 1. Contexto

Foi realizada uma análise técnica do mercado de transporte em Chicago com foco em:

- Estrutura competitiva  
- Concentração geográfica de demanda  
- Impacto das condições climáticas na eficiência operacional  

O objetivo foi identificar riscos operacionais e oportunidades estratégicas para uma possível entrada da Zuber no mercado.

---

## 2. Principais Achados

### Concentração de Mercado
- A Flash Cab lidera com 19.558 viagens no período analisado.
- O mercado apresenta forte concentração e barreiras de escala relevantes.

### Concentração Geográfica
- Loop e River North são os principais polos de destino.
- A demanda está fortemente centralizada nesses hubs financeiros.

### Impacto do Clima
- O teste estatístico confirmou diferença significativa na duração das viagens em dias chuvosos (p-value = 6,51e-12).
- Em condições adversas, o tempo médio das corridas aumenta de forma relevante.

---

## 3. Impacto Financeiro e Operacional

### Eficiência da Frota
- O aumento da duração média das viagens reduz o número de corridas por motorista por hora.
- Impacta diretamente a receita por hora e a margem operacional.

### Competitividade
- Alta concentração de mercado exige diferenciação operacional (tempo de espera, disponibilidade e gestão dinâmica de oferta).

### Alocação de Recursos
- Operação dispersa reduz eficiência.
- Foco em hubs estratégicos pode aumentar taxa de ocupação e reduzir ETA.

---

## 4. Recomendação

Recomenda-se avançar com uma abordagem estruturada e controlada:

1. **Implementar teste piloto de precificação dinâmica** em dias de clima adverso, para proteger margem e equilibrar oferta e demanda.
2. **Priorizar posicionamento estratégico da frota** nos hubs de maior demanda (Loop e River North), especialmente em horários comerciais.
3. **Monitorar indicadores-chave:** duração média, corridas por hora, receita por motorista e impacto na margem.

---

### Síntese Executiva

O mercado apresenta potencial relevante nos centros financeiros da cidade, porém exige eficiência operacional e mecanismos adaptativos para competir com players consolidados. A adoção de precificação dinâmica e alocação inteligente de frota aumenta a probabilidade de entrada sustentável e rentável.
